# Stable Diffusion：潜在扩散模型

这个 Notebook 使用预训练的 `Stable Diffusion v1.5` 展示潜在扩散模型（Latent Diffusion Model）的推理流程与架构解读。

内容包括：
- 文生图完整推理流程
- VAE Encoder/Decoder 编解码与重建
- 潜在空间（Latent Space）可视化
- CLIP 文本编码器输出分析
- Guidance Scale 与推理步数的效果对比
- 关键机制解读（Classifier-Free Guidance、Latent Diffusion、压缩比）
- 与 DDPM（Pixel-space）的对比

## 1. 环境准备

```bash
pip install diffusers transformers accelerate pillow
```

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO
from dataclasses import dataclass

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
# 无 GPU 时推理会较慢，建议使用 Colab T4 或本地 GPU

In [ ]:
from diffusers import StableDiffusionPipeline

# 加载预训练 SD v1.5，GPU 用 float16 节省显存，CPU 用 float32
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

pipe = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=dtype,
)
pipe = pipe.to(device)
# 关闭 NSFW 过滤，避免无意义的黑图
pipe.safety_checker = None
print('Pipeline loaded.')

## 2. Stable Diffusion 架构概述

Stable Diffusion 由三个核心组件组成：

| 组件 | 作用 |
|------|------|
| **VAE**（Variational Autoencoder）| 将 512×512 图像压缩到 64×64 潜在空间（压缩比 8×），再解码回图像 |
| **CLIP Text Encoder**（ViT-L/14）| 将文本 prompt 编码成 token 序列，引导去噪过程 |
| **U-Net**（Latent Diffusion U-Net）| 在潜在空间逐步预测并去除噪声，接受时间步和文本条件 |

推理流程：
1. CLIP 编码 prompt → text embeddings
2. 从纯高斯噪声开始（64×64×4 latent）
3. U-Net 在 latent 空间反复去噪（N 步）
4. VAE Decoder 把最终 latent 解码成 512×512 图像

In [ ]:
# 基础文生图示例
prompts = [
    'a golden retriever sitting in a sunlit park, photorealistic',
    'an astronaut riding a horse on the moon, digital art',
    'a cozy cabin in the snowy mountains at sunset, oil painting',
]

images = []
for prompt in prompts:
    with torch.autocast(device.type, dtype=dtype):
        image = pipe(prompt, num_inference_steps=20, guidance_scale=7.5).images[0]
    images.append(image)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, img, prompt in zip(axes, images, prompts):
    ax.imshow(img)
    ax.set_title(prompt[:50] + '...', fontsize=9)
    ax.axis('off')
plt.suptitle('Stable Diffusion Text-to-Image', fontsize=14)
plt.tight_layout()
plt.show()

## 3. VAE 编码与解码

In [ ]:
from torchvision import transforms

# 用第一张生成图像演示 VAE 编解码
img_pil = images[0]

# 预处理：归一化到 [-1, 1]
to_tensor = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])
img_tensor = to_tensor(img_pil).unsqueeze(0).to(device, dtype=dtype)

vae = pipe.vae
with torch.no_grad():
    # 编码到潜在空间
    latent = vae.encode(img_tensor).latent_dist.sample()
    # 缩放因子 0.18215 是 SD 的固定超参
    latent_scaled = latent * vae.config.scaling_factor
    # 解码回图像
    decoded = vae.decode(latent / vae.config.scaling_factor).sample

print(f'Original image  : {img_tensor.shape}')
print(f'Latent (scaled) : {latent_scaled.shape}   压缩比 = {512*512*3 / (64*64*4):.1f}x')
print(f'Reconstructed   : {decoded.shape}')

In [ ]:
def to_pil(tensor):
    # 把 [-1,1] 的 CHW tensor 转成可显示的 PIL
    img = (tensor.squeeze(0).float().cpu().clamp(-1, 1) + 1) / 2
    return transforms.ToPILImage()(img)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img_pil)
axes[0].set_title('Original (512×512)')
axes[0].axis('off')
axes[1].imshow(to_pil(decoded))
axes[1].set_title('VAE Reconstructed (512×512)')
axes[1].axis('off')
plt.suptitle('VAE Encode → Decode', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 可视化 latent 各通道（共 4 个通道）
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, ax in enumerate(axes):
    ch = latent_scaled[0, i].float().cpu().numpy()
    im = ax.imshow(ch, cmap='RdBu_r')
    ax.set_title(f'Latent channel {i}')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Latent Space Channels (64×64×4)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. CLIP 文本编码

In [ ]:
tokenizer    = pipe.tokenizer
text_encoder = pipe.text_encoder

test_prompts = [
    'a cat',
    'a golden retriever sitting in a sunlit park, photorealistic',
    'abstract colorful geometric shapes',
]

for p in test_prompts:
    tokens = tokenizer(p, return_tensors='pt', padding='max_length',
                       max_length=tokenizer.model_max_length, truncation=True)
    with torch.no_grad():
        emb = text_encoder(tokens.input_ids.to(device))[0]
    print(f'prompt: "{p[:40]}"')
    print(f'  token ids shape : {tokens.input_ids.shape}')
    print(f'  text embeddings : {emb.shape}   (seq_len=77, d_model=768)\n')

## 5. Guidance Scale 效果对比

In [ ]:
prompt  = 'a majestic lion, detailed fur, golden hour lighting'
scales  = [1.0, 7.5, 15.0]
gs_imgs = []

for gs in scales:
    generator = torch.Generator(device).manual_seed(42)
    with torch.autocast(device.type, dtype=dtype):
        img = pipe(prompt, guidance_scale=gs, num_inference_steps=20,
                   generator=generator).images[0]
    gs_imgs.append(img)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, img, gs in zip(axes, gs_imgs, scales):
    ax.imshow(img)
    ax.set_title(f'guidance_scale = {gs}')
    ax.axis('off')
plt.suptitle('Classifier-Free Guidance Scale 对比', fontsize=13)
plt.tight_layout()
plt.show()

## 6. 推理步数效果对比

In [ ]:
prompt = 'a serene Japanese garden with cherry blossoms'
steps  = [5, 20, 50]
step_imgs = []

for n in steps:
    generator = torch.Generator(device).manual_seed(0)
    with torch.autocast(device.type, dtype=dtype):
        img = pipe(prompt, num_inference_steps=n, guidance_scale=7.5,
                   generator=generator).images[0]
    step_imgs.append(img)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, img, n in zip(axes, step_imgs, steps):
    ax.imshow(img)
    ax.set_title(f'steps = {n}')
    ax.axis('off')
plt.suptitle('推理步数对比', fontsize=13)
plt.tight_layout()
plt.show()

## 7. 关键机制解读

### Classifier-Free Guidance（CFG）
- 训练时随机丢弃 prompt，让模型同时学有条件和无条件去噪。
- 推理时：`noise_pred = uncond + scale × (cond - uncond)`
- scale > 1 增强 prompt 引导力度；scale 过大会降低多样性甚至出现色彩过饱和。

### Latent Diffusion vs Pixel Diffusion（DDPM）
| | DDPM | Stable Diffusion |
|---|---|---|
| 扩散空间 | 原始像素（256×256×3） | 潜在空间（32×32×4） |
| 计算量 | 高 | 低（压缩 ~48×） |
| 图像质量 | 受限于分辨率 | 可生成高分辨率 |
| 条件控制 | 困难 | 灵活（文本/图像/深度等） |

### VAE 压缩比
- 512×512×3 → 64×64×4，面积压缩 64 倍，但 4 通道比 3 通道多，净压缩比 ~48×。
- 这使 U-Net 在低分辨率 latent 上去噪，大幅降低计算量。

### Scaling Factor（0.18215）
- SD 对 latent 施加缩放使其方差接近 1，保持与高斯噪声同量级，利于训练稳定。